In [ ]:
# -*- coding: utf-8 -*-
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
from scipy.interpolate import griddata
import os
import re
from tkinter import Tk, filedialog
from moviepy.editor import ImageSequenceClip

# --- Domain & time settings ---
X_MIN, X_MAX = 400, 1400
Y_MIN, Y_MAX = 0, 400
START_TIME = 40
END_TIME = 110
TIME_STEP = 1
SECONDS_PER_FRAME = 0.3

# --- Colormap range ---
Z_MIN, Z_MAX = -3, 2

# --- Figure layout ---
FIG_WIDTH = 4
FIG_HEIGHT = 2
FONT_NAME = "Times New Roman"
AXIS_FONT_SIZE = 6
TIME_LABEL_FONT_SIZE = 6


def pick_file(title):
    root = Tk()
    root.withdraw()
    root.update()
    file = filedialog.askopenfilename(title=title)
    root.destroy()
    return file


def parse_time_blocks(file):
    with open(file) as f:
        lines = f.readlines()
    times, blocks = [], []
    t, vals = None, []
    for line in lines:
        if "time=" in line:
            if t is not None:
                times.append(t)
                blocks.append(np.array(vals))
            t = float(re.findall(r"[-+]?\d*\.\d+|\d+", line)[0])
            vals = []
        else:
            try:
                vals.append(float(line.strip()))
            except:
                pass
    if t is not None:
        times.append(t)
        blocks.append(np.array(vals))
    return np.array(times), blocks


def parse_xy_grid(file):
    with open(file) as f:
        lines = f.readlines()
    times, x_blocks, y_blocks = [], [], []
    t, xs, ys = None, [], []
    for line in lines:
        line = line.strip()
        if "time=" in line:
            if t is not None:
                times.append(t)
                x_blocks.append(np.array(xs))
                y_blocks.append(np.array(ys))
            t = float(re.findall(r"[-+]?\d*\.\d+|\d+", line)[0])
            xs, ys = [], []
        else:
            try:
                parts = line.split(",")
                xs.append(float(parts[0].strip()))
                ys.append(float(parts[1].strip()))
            except:
                pass
    if t is not None:
        times.append(t)
        x_blocks.append(np.array(xs))
        y_blocks.append(np.array(ys))
    return np.array(times), x_blocks, y_blocks


def nearest_index(times, target):
    return np.argmin(np.abs(times - target))


def create_colormap():
    return LinearSegmentedColormap.from_list(
        "strain_map",
        ["#1b3cff", "#00bfff", "#40e0d0", "#00cc66",
         "#66ff33", "#ccff00", "#ffff00", "#ffcc00"],
        N=256
    )


def plot_frame(x, y, z, time_val, out_path):
    n = min(len(x), len(y), len(z))
    x, y, z = x[:n], y[:n], z[:n]

    mask = (x >= X_MIN) & (x <= X_MAX) & (y >= Y_MIN) & (y <= Y_MAX)
    x, y, z = x[mask], y[mask], z[mask]

    xi = np.linspace(X_MIN, X_MAX, 401)
    yi = np.linspace(Y_MIN, Y_MAX, 301)
    Xi, Yi = np.meshgrid(xi, yi)
    Zi = griddata((x, y), z, (Xi, Yi), method='linear')

    plt.rcParams['font.family'] = FONT_NAME
    fig, ax = plt.subplots(figsize=(FIG_WIDTH, FIG_HEIGHT), dpi=300)
    fig.subplots_adjust(left=0.10, right=1.0, bottom=0.22, top=0.92)

    norm = Normalize(vmin=Z_MIN, vmax=Z_MAX)
    im = ax.imshow(
        Zi,
        extent=[X_MIN, X_MAX, Y_MIN, Y_MAX],
        origin='lower',
        cmap=create_colormap(),
        norm=norm,
        aspect='auto',
        interpolation='nearest'
    )

    ax.set_xlim(X_MIN, X_MAX)
    ax.set_ylim(Y_MAX, Y_MIN)
    ax.set_xlabel("Distance, km", fontsize=AXIS_FONT_SIZE)
    ax.set_ylabel("Depth, km", fontsize=AXIS_FONT_SIZE)
    ax.set_xticks(np.arange(X_MIN, X_MAX + 1, 100))
    ax.set_yticks(np.arange(Y_MIN, Y_MAX + 1, 50))
    ax.tick_params(axis='both', labelsize=AXIS_FONT_SIZE)
    for spine in ax.spines.values():
        spine.set_linewidth(0.8)
        spine.set_color("black")

    cbar = fig.colorbar(im, ax=ax, fraction=0.12, pad=0.005)
    cbar.ax.set_anchor('W')
    cbar.set_label(r'$\log_{10}$(bulk strain)', fontsize=AXIS_FONT_SIZE, fontname=FONT_NAME)
    cbar.ax.tick_params(labelsize=AXIS_FONT_SIZE)
    for tick in cbar.ax.get_yticklabels():
        tick.set_fontname(FONT_NAME)
        tick.set_fontsize(AXIS_FONT_SIZE)

    ax.text(
        0.98, 0.03,
        f"{int(time_val // 10) * 10} Myr",
        transform=ax.transAxes,
        fontsize=TIME_LABEL_FONT_SIZE,
        ha='right', va='bottom',
        bbox=dict(facecolor='white', edgecolor='black', linewidth=0.5, pad=0.15)
    )

    plt.savefig(out_path, dpi=300)
    plt.close()


def main():
    print("Select input files...")
    strain_file = pick_file("Select bulk strain file")
    xy_file = pick_file("Select XY grid file")

    out_dir = os.path.dirname(strain_file)
    os.makedirs(out_dir, exist_ok=True)

    t_s, s_blocks = parse_time_blocks(strain_file)
    t_xy, x_blocks, y_blocks = parse_xy_grid(xy_file)

    frames = []
    for i in range(len(t_s)):
        if i % TIME_STEP != 0:
            continue
        t = t_s[i]
        if t < START_TIME or t > END_TIME:
            continue
        ixy = nearest_index(t_xy, t)
        out_path = os.path.join(out_dir, f"frame_{i:04d}.png")
        plot_frame(x_blocks[ixy], y_blocks[ixy], s_blocks[i], t, out_path)
        frames.append(out_path)
        print(f"  Saved frame {i} (t={t:.2f})")

    frames = [f for f in frames if os.path.exists(f) and os.path.getsize(f) > 0]
    if not frames:
        print("No frames were generated. Check time range and input files.")
        return

    print(f"\nRendering video from {len(frames)} frames...")
    fps = 1 / SECONDS_PER_FRAME
    clip = ImageSequenceClip(frames, fps=fps)
    clip.write_videofile(
        os.path.join(out_dir, "bulk_strain.mp4"),
        codec="libx264",
        fps=fps,
        audio=False
    )
    print("Done. Output saved to bulk_strain.mp4")


if __name__ == "__main__":
    main()